In [ ]:
import pandas as pd

In [ ]:
# Load dataset from Feature Selected csv
df = pd.read_csv("./datasets/FeatureSelectedDataset.csv")
#df = pd.read_csv("datasets/TestDatasetExample.csv",  index_col=False)
#df = pd.read_csv("datasets/TrainDataset2025.csv",  index_col=False)
#df = pd.read_csv("datasets/FinalTestDataset2025", index_col = False)

df.info()

# Machine Learning Method Development

### Pre-Method Processing

1. Verify dataset
2. Sample Dataset
3. Separate Features from Outcomes
4. Train-Test(-Validation) Split

#### 1. Verify Dataset

In [ ]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Columns: 121 entries, ID to original_ngtdm_Strength
dtypes: float64(108), int64(12), object(1)
memory usage: 378.3+ KB


,pCR (outcome),RelapseFreeSurvival (outcome),Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
count,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.00000,...,4.000000e+02,4.000000e+02,4.000000e+02,400.000000,4.000000e+02,400.000000,400.000000,400.000000,400.000000,400.000000
mean,12.697500,56.000208,51.804674,0.547500,2.902500,2.797500,2.830000,9.875000,6.562500,8.63250,...,3.957637e-01,3.911005e-01,2.722189e+00,0.003347,5.679717e+07,178.311246,32500.032620,0.056935,0.005965,0.029322
std,111.107417,27.137584,10.948522,0.498362,49.932114,49.937068,49.935558,86.092911,70.444284,86.20034,...,1.666319e-01,1.615922e-01,7.648849e-01,0.002419,7.063846e+08,1045.453432,177545.921568,0.047179,0.008379,0.115915
min,0.000000,0.000000,23.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.00000,...,7.050000e-11,7.050000e-11,-3.200000e-16,0.000008,0.000000e+00,0.000000,0.000248,0.000000,0.000000,0.000000
25%,0.000000,38.000000,44.516769,0.000000,0.000000,0.000000,0.000000,2.000000,1.000000,1.00000,...,3.199017e-01,3.184398e-01,2.340783e+00,0.001389,1.030473e+06,18.760570,0.001826,0.018628,0.000310,0.001464
50%,0.000000,55.000000,51.019507,1.000000,0.000000,0.000000,0.000000,2.000000,1.000000,1.00000,...,4.095627e-01,4.054695e-01,2.814884e+00,0.002944,3.277334e+06,67.929659,0.004383,0.047740,0.002330,0.003276
75%,0.000000,73.000000,60.000000,1.000000,1.000000,1.000000,1.000000,3.000000,2.000000,1.00000,...,5.000049e-01,4.956920e-01,3.304411e+00,0.004798,9.079686e+06,157.370294,0.013769,0.085321,0.007962,0.009479
max,999.000000,144.000000,79.603012,1.000000,999.000000,999.000000,999.000000,999.000000,999.000000,999.00000,...,8.773779e-01,8.571429e-01,4.947427e+00,0.011301,1.390001e+10,20764.693790,1000000.000000,0.285100,0.060742,1.145601


#### 2. Sample Dataset
Generative Models:
- Variational Autoencoder (VAE) - Used in Dimensionality Reduction
- Generative Adversarial Network (GAN):#####

These do ____

##### VAE

##### GAN
A GAN is used to generate more X_train and y_train data points (synthetic data) if the original dataset was too small. This is done by training a Generator to output realistic (X, y) pairs that fool a Discriminator.

Here, given that the Example dataset only had three samples, it was deemed beneficial to use.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.preprocessing import MinMaxScaler
import numpy as np

In [ ]:
# Preprocessing

ids = df['ID']
dff_names = dff.columns


# Normalize Data to [0, 1] for GAN training (common practice with sigmoid output)
scaler = MinMaxScaler(feature_range=(-1, 1)) # Tanh activation range
X_train_scaled = scaler.fit_transform(dff)

# Categorical columns to be rounded later
# Based on common medical dataset knowledge and inspection of values (integers vs floats)
categorical_cols = ['ER', 'PgR', 'HER2', 'TrippleNegative', 'ChemoGrade', 'Proliferation', 'HistologyType', 'LNStatus', 'TumourStage', 'Gene']

In [ ]:
# Building GAN

latent_dim = 64 # Size of random noise vector
data_dim = X_train_scaled.shape[1]

# Generator
def build_generator(latent_dim, data_dim):
    model = tf.keras.Sequential()
    model.add(layers.Dense(128, input_dim=latent_dim))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.Dense(256))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.Dense(data_dim, activation='tanh')) # Output scaled between -1 and 1
    return model

# Discriminator
def build_discriminator(data_dim):
    model = tf.keras.Sequential()
    model.add(layers.Dense(256, input_dim=data_dim))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.4))
    model.add(layers.Dense(128))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.4))
    model.add(layers.Dense(1, activation='sigmoid')) # Output probability
    return model

generator = build_generator(latent_dim, data_dim)
discriminator = build_discriminator(data_dim)

# Compile Discriminator
optimizer = optimizers.Adam(0.0002, 0.5)
discriminator.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Combined Model (Stack Generator and Discriminator)
# Train Generator to fool Discriminator
discriminator.trainable = False
z = layers.Input(shape=(latent_dim,))
img = generator(z)
validity = discriminator(img)
combined = models.Model(z, validity)
combined.compile(loss='binary_crossentropy', optimizer=optimizer)

In [ ]:
# Train GAN
epochs = 500 
batch_size = 3 
# Note: Batch size is tiny because dataset is tiny. 
# In real scenarios, batch_size would be 32 or 64.

valid = np.ones((batch_size, 1))
fake = np.zeros((batch_size, 1))

for epoch in range(epochs):
    # Train Discriminator 
    # Select a random batch of images (here, just the whole dataset essentially)
    idx = np.random.randint(0, X_train_scaled.shape[0], batch_size)
    real_data = X_train_scaled[idx]

    # Generate noise and fake data
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    gen_data = generator.predict(noise, verbose=0)

    # Train
    d_loss_real = discriminator.train_on_batch(real_data, valid)
    d_loss_fake = discriminator.train_on_batch(gen_data, fake)
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

    # Train Generator
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    g_loss = combined.train_on_batch(noise, valid)

    if epoch % 100 == 0:
        print(f"{epoch} [D loss: {d_loss[0]}, acc.: {100*d_loss[1]}%] [G loss: {g_loss}]")

In [ ]:
# Generate New Data
num_new_samples = 5
noise = np.random.normal(0, 1, (num_new_samples, latent_dim))
gen_data_scaled = generator.predict(noise)

# 6. Post-process
gen_data = scaler.inverse_transform(gen_data_scaled)
gen_df = pd.DataFrame(gen_data, columns= dff_names)

# Round categorical columns to nearest integer
for col in categorical_cols:
    if col in gen_df.columns:
        gen_df[col] = gen_df[col].round().astype(int)

# Generate new IDs
last_id_num = int(ids.iloc[-1].replace("TRG", ""))
new_ids = [f"TRG{str(last_id_num + i + 1).zfill(6)}" for i in range(num_new_samples)]
gen_df.insert(0, 'ID', new_ids)

# Merge with original
final_df = pd.concat([df, gen_df], ignore_index=True)
    
print(final_df.head(10))

VAE-GAN
#- Possibly try VAE-GAN to remove issue with gaussian distribution

#### 3. Separate Features from Outcomes
(data is from csv)

In [ ]:
# Separating features and outcomes

dff = df.drop('ID', axis=1)
X = dff.drop(['pCR (outcome)', 'RelapseFreeSurvival (outcome)'], axis=1)
y_class = dff['pCR (outcome)']
y_reg = dff['RelapseFreeSurvival (outcome)']

#### 4. Train-test-Validation split
where do we do cross validation, with what methods to generate error

In [ ]:
# Train test split
from sklearn.model_selection import train_test_split

# Split Data
X_train, X_test, y_train, y_test, y_clf_train, y_clf_test = train_test_split(X, y_reg, y_class, test_size=0.2, random_state = 42, stratify= y_class)


### Machine Learning Models

##### Linear:
- Multivariate Linear Regression
- ~~Logistic Regression~~ (Took over 10 min until interrupted) | Classification
- Naïve Bayes (don't forget different types) | Classification
- K-Nearest Neighbour (KNN) | Classification
- Support Vector Classifier/Regressor

##### Non-Linear:
- Kernel Support Vector Classifier/Regressior
- Decision Tree Classifier/Regressor
- Random Forest Classifier/Regressor

##### Deep Learning Models:
- Multilayer Perceptron Classifier/Regressor
- Convolution Neural Networks -------------------------------------------------

### Classification

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


results = {}

##### Logistic Regression

In [26]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression
log_reg = LogisticRegression(max_iter=100)
log_reg.fit(X_train, y_clf_train)
results['Logistic Regression Acc'] = log_reg.score(X_test, y_clf_test)

C:\Users\EK\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


##### Naïve Bayes
Different types: gaussian, bernoulli, (counts), (another one I forgot)

In [7]:
# Naïve Bayes (Gaussian)
nb = GaussianNB()
nb.fit(X_train, y_clf_train)
results['Naive Bayes Acc'] = nb.score(X_test, y_clf_test)

##### K Nearest Neighbour
Value of K hyperparameter

In [8]:
# KNN
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_clf_train)
results['KNN Acc'] = knn.score(X_test, y_clf_test)

##### Support Vector Classifier
Hard Margin, soft margin

In [ ]:
# Hard Margin SVM (Approximated with large C)
svm_hard = SVC(kernel='linear', C=100)
svm_hard.fit(X_train, y_clf_train)
results['Hard Margin SVM Acc'] = svm_hard.score(X_test, y_clf_test)

##### Kernel SVC

In [9]:
# Kernel SVM (RBF)
svm_kernel = SVC(kernel='rbf')
svm_kernel.fit(X_train, y_clf_train)
results['Kernel SVM Acc'] = svm_kernel.score(X_test, y_clf_test)

##### Decision Tree
Reasons for params

In [10]:
# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_clf_train)
results['Decision Tree Acc'] = dt.score(X_test, y_clf_test)

##### Random Forest
Reasons for params

In [11]:
# Random Forest
rf = RandomForestClassifier(n_estimators=10)
rf.fit(X_train, y_clf_train)
results['Random Forest Acc'] = rf.score(X_test, y_clf_test)

##### Multi-Layer Perceptron Classifier
Explain Parameters

In [12]:
# MLP
mlp = MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=500)
mlp.fit(X_train, y_clf_train)
results['MLP Acc'] = mlp.score(X_test, y_clf_test)

In [22]:
print("Model execution successful. Results on dummy test set:")
print(results)

Model execution successful. Results on dummy test set:
{'Naive Bayes Acc': 0.1375, 'KNN Acc': 0.75, 'Kernel SVM Acc': 0.775, 'Decision Tree Acc': 0.6875, 'Random Forest Acc': 0.7625, 'MLP Acc': 0.775, 'Logistic Regression Acc': 0.7625}


### Regression

##### Multivariate Linear Regression
explain params

In [14]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
print(f"Linear Regression Score: {lin_reg.score(X_test, y_test)}")

Linear Regression Score: -2254.2453237337036


##### Support Vector Regressor
Explain parameters

In [15]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

# Kernel SVR (RBF)
svr = SVR(kernel='rbf')
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)

print(f"SVR MSE: {mean_squared_error(y_test, y_pred_svr)}")

SVR MSE: 677.3637249391907


##### Kernel SVR

##### Decision Tree Regressor
Explain Parameters

In [16]:
from sklearn.tree import DecisionTreeRegressor

dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(X_train, y_train)
y_pred_dt = dt_reg.predict(X_test)

print(f"Decision Tree MSE: {mean_squared_error(y_test, y_pred_dt)}")

Decision Tree MSE: 1985.0733506779725


##### Random Forest Regressor
Explain Parameters

In [17]:
from sklearn.ensemble import RandomForestRegressor

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)
y_pred_rf = rf_reg.predict(X_test)

print(f"Random Forest MSE: {mean_squared_error(y_test, y_pred_rf)}")

Random Forest MSE: 710.6563212945834


##### Multi Layer Perceptron Regressor
Explain Parameters

In [18]:
from sklearn.neural_network import MLPRegressor

# MLP with 2 hidden layers
mlp_reg = MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
mlp_reg.fit(X_train, y_train)
y_pred_mlp = mlp_reg.predict(X_test)

print(f"MLP MSE: {mean_squared_error(y_test, y_pred_mlp)}")

MLP MSE: 103234667229.52676


# K Fold Cross Validation
Create pipeline and use k fold 

# Method Evaluation

### Classification
- Confusion Matrix
- Precision/Recall/Specificity/Sensitivity
- (Balanced) Classification Accuracy

### Regression
- Mean Square Error
- etc

In [ ]:
# Export chosen Model (using pickle?)
# Go to "final notebooks"